In [11]:
import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

np.random.seed(42)

PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"
MODELS_DIR    = r"C:\Project_EquiGuard\outputs\models"
ARRAYS_DIR    = os.path.join(PROCESSED_DIR, "arrays")
LOGS_DIR      = r"C:\Project_EquiGuard\logs"

os.makedirs(LOGS_DIR, exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [12]:
# Load scaled datasets

X_train = np.load(os.path.join(ARRAYS_DIR, "X_train_scaled.npy"))
y_train = np.load(os.path.join(ARRAYS_DIR, "y_train.npy"))
X_val   = np.load(os.path.join(ARRAYS_DIR, "X_val_scaled.npy"))
y_val   = np.load(os.path.join(ARRAYS_DIR, "y_val.npy"))

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"\ny_train: {np.unique(y_train, return_counts=True)}")
print(f"y_val:   {np.unique(y_val,   return_counts=True)}")

X_train: (210, 29)
X_val:   (132, 29)

y_train: (array([0, 1]), array([ 42, 168]))
y_val:   (array([0, 1]), array([ 10, 122]))


In [20]:
# Compute class imbalance ratio for XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Corrected scale_pos_weight: {scale_pos:.4f}")

# Candidate models were evaluated on the validation set.
# Random Forest achieved the strongest overall performance
# and was retained as the final model.
final_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_val)
y_prob_final = final_model.predict_proba(X_val)[:, 1]

print("\nRANDOM FOREST FINAL MODEL : VALIDATION SET")
print(classification_report(y_val, y_pred_final,
      target_names=["Sound", "Lame"]))
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_final):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_final))

# Save final model
joblib.dump(final_model, os.path.join(MODELS_DIR, "final_model.pkl"))
print(f"\nFinal model saved.")

# Feature importance analysis
feature_names_final = joblib.load(
    os.path.join(MODELS_DIR, "feature_names_final.pkl")
)
importances = final_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:10]

print("\n TOP 10 FEATURE IMPORTANCES ")
for i in top_idx:
    print(f"{feature_names_final[i]:<30} {importances[i]:.4f}")

Corrected scale_pos_weight: 0.2500

RANDOM FOREST FINAL MODEL : VALIDATION SET
              precision    recall  f1-score   support

       Sound       0.38      0.30      0.33        10
        Lame       0.94      0.96      0.95       122

    accuracy                           0.91       132
   macro avg       0.66      0.63      0.64       132
weighted avg       0.90      0.91      0.90       132

ROC-AUC: 0.8430

Confusion Matrix:
[[  3   7]
 [  5 117]]

Final model saved.

 TOP 10 FEATURE IMPORTANCES 
mahalanobis_score              0.5346
MnDHead                        0.1896
MnDSac                         0.1615
MxDHead                        0.0169
UpDHead                        0.0148
HHD                            0.0143
MxDSac                         0.0141
UpDSac                         0.0136
MxDisLFasym                    0.0033
MnDisLFasym                    0.0033


In [14]:
# Define base models

xgb = XGBClassifier(
    n_estimators=200,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric="logloss",
    verbosity=0
)

lgbm = LGBMClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    verbose=-1
)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

svm = SVC(
    kernel="rbf",
    class_weight="balanced",
    probability=True,
    random_state=42
)

print("Base models defined:")
print("  XGBoost, LightGBM, RandomForest, SVM")

Base models defined:
  XGBoost, LightGBM, RandomForest, SVM


In [15]:
# Build stacking ensemble

meta_learner = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

stacking = StackingClassifier(
    estimators=[
        ("xgb",  xgb),
        ("lgbm", lgbm),
        ("rf",   rf),
        ("svm",  svm)
    ],
    final_estimator=meta_learner,
    cv=5,
    passthrough=False,
    n_jobs=-1
)

print("Stacking ensemble built.")
print("Meta learner: Logistic Regression")
print("CV folds: 5")

Stacking ensemble built.
Meta learner: Logistic Regression
CV folds: 5


In [16]:
#Train stacking ensemble

print("Training stacking ensemble...")

stacking.fit(X_train, y_train)

print("\nTraining complete.")

joblib.dump(stacking, os.path.join(MODELS_DIR, "stacking_model.pkl"))
print("Stacking model saved.")

Training stacking ensemble...

Training complete.
Model saved to: C:\Project_EquiGuard\outputs\models/stacking_model.pkl


In [17]:
# Evaluate on validation set

y_pred  = stacking.predict(X_val)
y_prob  = stacking.predict_proba(X_val)[:, 1]

print(" STACKING ENSEMBLE : VALIDATION SET ")
print(classification_report(y_val, y_pred, 
      target_names=["Sound", "Lame"]))
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

=== STACKING ENSEMBLE — VALIDATION SET ===
              precision    recall  f1-score   support

       Sound       0.23      0.30      0.26        10
        Lame       0.94      0.92      0.93       122

    accuracy                           0.87       132
   macro avg       0.59      0.61      0.60       132
weighted avg       0.89      0.87      0.88       132

ROC-AUC: 0.8344

Confusion Matrix:
[[  3   7]
 [ 10 112]]


C:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [18]:
# Ablation study
# Evaluate the contribution of the Mahalanobis feature and compare ensemble performance against Random Forest.

feature_names = joblib.load(
    os.path.join(MODELS_DIR, "feature_names_final.pkl")
)
mahal_idx = feature_names.index("mahalanobis_score")

results = {}

# Random Forest alone
rf_solo = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42
)
rf_solo.fit(X_train, y_train)
results["RF alone"] = f1_score(y_val, rf_solo.predict(X_val))

# Stacking without Mahalanobis feature
X_train_no_mahal = np.delete(X_train, mahal_idx, axis=1)
X_val_no_mahal   = np.delete(X_val,   mahal_idx, axis=1)

stacking_no_mahal = StackingClassifier(
    estimators=[
        ("xgb",  XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos,
                               random_state=42, eval_metric="logloss", verbosity=0)),
        ("lgbm", LGBMClassifier(n_estimators=200, class_weight="balanced",
                                random_state=42, verbose=-1)),
        ("rf",   RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                        random_state=42)),
        ("svm",  SVC(kernel="rbf", class_weight="balanced",
                     probability=True, random_state=42))
    ],
    final_estimator=LogisticRegression(class_weight="balanced",
                                       random_state=42, max_iter=1000),
    cv=5, n_jobs=-1
)
stacking_no_mahal.fit(X_train_no_mahal, y_train)
results["Stacking without Mahalanobis"] = f1_score(
    y_val, stacking_no_mahal.predict(X_val_no_mahal)
)

# Full stacking with Mahalanobis (already trained)
results["Stacking with Mahalanobis"] = f1_score(y_val, y_pred)

print(" ABLATION STUDY : VALIDATION F1 ")
for name, score in results.items():
    print(f"{name:<35} F1: {score:.4f}")

 ABLATION STUDY : VALIDATION F1 
RF alone                            F1: 0.9516
Stacking without Mahalanobis        F1: 0.7980
Stacking with Mahalanobis           F1: 0.9295


C:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
# Log results

log_path = os.path.join(LOGS_DIR, "experiment_log.txt")

with open(log_path, "a") as f:
    f.write("\n PHASE 5 RESULTS\n")
    f.write(f"X_train shape: {X_train.shape}\n")
    f.write(f"X_val shape:   {X_val.shape}\n\n")
    f.write("VALIDATION METRICS:\n")
    f.write(classification_report(y_val, y_pred,
            target_names=["Sound", "Lame"]))
    f.write(f"ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}\n\n")
    f.write("ABLATION RESULTS:\n")
    for name, score in results.items():
        f.write(f"{name}: {score:.4f}\n")

print(f"Results logged to: {log_path}")

Results logged to: C:\Project_EquiGuard\logs\experiment_log.txt
